# Assignment 10

This notebook solves Mid-Semester questions **2, 4, and 5**, then fine-tunes a base multilingual BERT model for the Assignment 1 language-identification task. The BERT run is opt-in because downloading the model and training 23 classes can take several minutes.

## Mid-Semester Question 2 — Beam search

Use log probabilities only for ranking; cumulative probabilities below are shown in ordinary probability space.

| step | expanded hypotheses (cumulative probability) | retained beam (k = 2) |
|---|---|---|
| 1 | `cat` = 0.5, `sat` = 0.3, `mat` = 0.2 | `cat` (0.5), `sat` (0.3) |
| 2 | `cat sat` = 0.5×0.6 = **0.30**; `cat mat` = 0.5×0.3 = **0.15**; `cat </s>` = 0.5×0.1 = **0.05**; `sat cat` = 0.3×0.2 = **0.06**; `sat mat` = 0.3×0.7 = **0.21**; `sat </s>` = 0.3×0.1 = **0.03** | `cat sat` (0.30), `sat mat` (0.21) |
| 3 | `cat sat mat` = 0.30×0.8 = **0.24**; `cat sat </s>` = 0.30×0.2 = **0.06**; `sat mat cat` = 0.21×0.1 = **0.021**; `sat mat </s>` = 0.21×0.9 = **0.189** | `cat sat mat` (0.24), `sat mat </s>` (0.189) |

Therefore the highest-probability final output is **`cat sat mat`** with probability 0.24. The other retained hypothesis ends at `</s>` and represents `sat mat` with probability 0.189. We ignore `P(cat|mat)` and `P(sat|mat)` because the vocabulary supplies no probabilities for a `mat` prefix at time 2 and, more importantly, `mat` was not retained in the beam after time 1. Beam search expands only retained hypotheses, not every hypothesis in the full search tree.

## Mid-Semester Question 4 — BPE for two merges

Represent each word as characters followed by `</w>`, and count each word with its corpus frequency. The initial vocabulary is:

```text
low: 5, lower: 2, lowest: 3, new: 2, newest: 1, widest: 3
```

The most frequent adjacent pair is `(l, o)` with count 10. Merge 1 gives `lo`. Recounting makes `(lo, w)` the most frequent pair, also with count 10. Merge 2 gives `low`.

After the two merges, the vocabulary is:

```text
low </w>                 (5)
low e r </w>              (2)
low e s t </w>            (3)
n e w </w>                (2)
n e w e s t </w>          (1)
w i d e s t </w>          (3)
```

Merge rules: `(l, o) -> lo`, then `(lo, w) -> low`. BPE handles an unseen word by applying the learned merges greedily to its characters. Any characters or character pieces not covered by the vocabulary are represented by an unknown token, so the whole word does not need to be in the vocabulary; this is why BPE greatly reduces out-of-vocabulary failures.

In [1]:
from collections import Counter
corpus = "low lower low lower lowest low lowest low lowest low new newest widest widest new widest".split()
vocab = Counter(tuple(list(word) + ["</w>"]) for word in corpus)
merges = []
for _ in range(2):
    pair_counts = Counter()
    for symbols, frequency in vocab.items():
        pair_counts.update({symbols[i:i + 2]: frequency for i in range(len(symbols) - 1)})
    pair = max(pair_counts, key=lambda item: pair_counts[item])
    merges.append((pair, pair_counts[pair]))
    updated = Counter()
    for symbols, frequency in vocab.items():
        result, index = [], 0
        while index < len(symbols):
            if index + 1 < len(symbols) and symbols[index:index + 2] == pair:
                result.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                result.append(symbols[index])
                index += 1
        updated[tuple(result)] += frequency
    vocab = updated
print("merges:", merges)
print("vocabulary:", dict(vocab))

merges: [(('l', 'o'), 10), (('lo', 'w'), 10)]
vocabulary: {('low', '</w>'): 5, ('low', 'e', 'r', '</w>'): 2, ('low', 'e', 's', 't', '</w>'): 3, ('n', 'e', 'w', '</w>'): 2, ('n', 'e', 'w', 'e', 's', 't', '</w>'): 1, ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3}


## Mid-Semester Question 5 — TF-IDF, class vectors, and Doc2Vec

The vocabulary is `[machine, learning, is, useful, powerful, language, processing]`. With raw term frequency and `idf(t) = ln(N/df(t))`, `N = 4`, the document frequencies are `df(machine)=df(powerful)=df(processing)=1`, `df(language)=2`, `df(learning)=df(useful)=3`, and `df(is)=4`.

| word | IDF | D1 | D2 | D3 | D4 |
|---|---:|---:|---:|---:|---:|
| machine | 1.3863 | 1.3863 | 0 | 0 | 0 |
| learning | 0.2877 | 0.2877 | 0.2877 | 0.2877 | 0 |
| is | 0 | 0 | 0 | 0 | 0 |
| useful | 0.2877 | 0.2877 | 0 | 0.2877 | 0.2877 |
| powerful | 1.3863 | 0 | 1.3863 | 0 | 0 |
| language | 0.6931 | 0 | 0 | 0.6931 | 0.6931 |
| processing | 1.3863 | 0 | 0 | 0 | 1.3863 |

Each document is represented by its seven-dimensional TF-IDF row. A simple class representation is the centroid (mean) of the Technology rows D1 and D2 or the Language rows D3 and D4; a new document can be assigned to the nearest centroid using cosine similarity. `is` has zero IDF because it occurs in all documents and contributes no class evidence.

For Doc2Vec, use **Distributed Memory (PV-DM)**: concatenate or average the document vector and context-word vectors, feed them through a neural layer, and predict the missing/target word with softmax. Each document has a learned ID vector, so the document representation is not merely an average of word vectors. An alternative is Distributed Bag of Words (PV-DBOW), where the document ID alone predicts words sampled from the document.

In [2]:
import math
import pandas as pd

documents = {
    "D1": "machine learning is useful",
    "D2": "machine learning is powerful",
    "D3": "learning language is useful",
    "D4": "language processing is useful",
}
tokens = {name: text.split() for name, text in documents.items()}
vocabulary = sorted(set(word for words in tokens.values() for word in words))
idf = {word: math.log(len(tokens) / sum(word in words for words in tokens.values())) for word in vocabulary}
tfidf = pd.DataFrame(
    [[words.count(word) * idf[word] for word in vocabulary] for words in tokens.values()],
    index=tokens,
    columns=vocabulary,
)
print("vocabulary:", vocabulary)
display(tfidf.round(4))
print("Technology centroid:\n", tfidf.loc[["D1", "D2"]].mean())
print("Language centroid:\n", tfidf.loc[["D3", "D4"]].mean())

vocabulary: ['is', 'language', 'learning', 'machine', 'powerful', 'processing', 'useful']


,is,language,learning,machine,powerful,processing,useful
D1,0.0,0.0000,0.2877,0.6931,0.0000,0.0000,0.2877
D2,0.0,0.0000,0.2877,0.6931,1.3863,0.0000,0.0000
D3,0.0,0.6931,0.2877,0.0000,0.0000,0.0000,0.2877
D4,0.0,0.6931,0.0000,0.0000,0.0000,1.3863,0.2877


Technology centroid:
 is            0.000000
language      0.000000
learning      0.287682
machine       0.693147
powerful      0.693147
processing    0.000000
useful        0.143841
dtype: float64
Language centroid:
 is            0.000000
language      0.693147
learning      0.143841
machine       0.000000
powerful      0.000000
processing    0.693147
useful        0.287682
dtype: float64


## BERT language identification

The labels and the 80:10:10 stratified splits are loaded unchanged from Assignment 1. mBERT is a suitable base model because its WordPiece vocabulary contains multilingual subwords and it supports the Indic scripts represented by the 23 labels. The comparison values below are the test macro-F1 values reported by Assignment 6; BERT is added only after the opt-in run.

In [3]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

SEED = 42
RUN_BERT = False
MODEL_NAME = "google-bert/bert-base-multilingual-cased"
MAX_LENGTH = 128
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd()
if ROOT.name != "Assignment - 10":
    ROOT = ROOT / "Assignment - 10"
DATA = ROOT.parent / "Assignment - 1" / "data"
with (DATA / "labels.json").open(encoding="utf-8") as handle:
    labels = json.load(handle)
label_to_id = {label: index for index, label in enumerate(labels)}

def read_split(path):
    frame = pd.read_csv(path, sep="\t")
    frame["label_id"] = frame["label"].map(label_to_id)
    if frame["label_id"].isna().any():
        raise ValueError(f"Unknown label in {path}")
    return frame

train = read_split(DATA / "train.tsv")
validation = read_split(DATA / "val.tsv")
test = read_split(DATA / "test.tsv")
print({name: len(frame) for name, frame in [("train", train), ("validation", validation), ("test", test)]})
print("classes:", len(labels))

{'train': 16995, 'validation': 2279, 'test': 2052}
classes: 23


In [4]:
comparison = pd.DataFrame([
    ("Logistic regression", 0.8451),
    ("MLP", 0.8591),
    ("LSTM", 0.8789),
    ("GRU", 0.8841),
    ("FastText + wiki", 0.9479),
], columns=["model", "test_macro_f1"])

class TextDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.texts = frame["text"].tolist()
        self.labels = torch.tensor(frame["label_id"].astype(int).to_numpy(), dtype=torch.long)
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        encoded = self.tokenizer(self.texts[index], truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
        return {key: value.squeeze(0) for key, value in encoded.items()} | {"labels": self.labels[index]}

def macro_f1(y_true, y_pred, class_count):
    scores = []
    for cls in range(class_count):
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(scores))

if RUN_BERT:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label_to_id)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    loaders = {name: DataLoader(TextDataset(frame, tokenizer), batch_size=BATCH_SIZE, shuffle=name == "train") for name, frame in [("train", train), ("validation", validation), ("test", test)]}
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    best_f1, best_state = -1.0, None
    for epoch in range(EPOCHS):
        model.train()
        for batch in loaders["train"]:
            batch = {key: value.to(device) for key, value in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        model.eval()
        true, predicted = [], []
        with torch.no_grad():
            for batch in loaders["validation"]:
                labels_batch = batch.pop("labels").numpy()
                logits = model(**{key: value.to(device) for key, value in batch.items()}).logits.cpu().numpy()
                true.extend(labels_batch)
                predicted.extend(logits.argmax(axis=1))
        score = macro_f1(np.array(true), np.array(predicted), len(labels))
        print(f"epoch {epoch + 1}: validation macro-F1={score:.4f}")
        if score > best_f1:
            best_f1, best_state = score, {key: value.cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    true, predicted = [], []
    with torch.no_grad():
        for batch in loaders["test"]:
            labels_batch = batch.pop("labels").numpy()
            logits = model(**{key: value.to(device) for key, value in batch.items()}).logits.cpu().numpy()
            true.extend(labels_batch)
            predicted.extend(logits.argmax(axis=1))
    y_true, y_pred = np.array(true), np.array(predicted)
    bert_f1 = macro_f1(y_true, y_pred, len(labels))
    accuracy = float(np.mean(y_true == y_pred))
    comparison.loc[len(comparison)] = ["mBERT fine-tuned", bert_f1]
    print(f"mBERT test macro-F1={bert_f1:.4f}; accuracy={accuracy:.4f}")
else:
    print("RUN_BERT is False; set it to True to fine-tune mBERT and add its measured result.")

display(comparison.sort_values("test_macro_f1", ascending=False).reset_index(drop=True))

RUN_BERT is False; set it to True to fine-tune mBERT and add its measured result.


,model,test_macro_f1
0,FastText + wiki,0.9479
1,GRU,0.8841
2,LSTM,0.8789
3,MLP,0.8591
4,Logistic regression,0.8451
